## final_project_part_1.ipynb
### DSA4323/5323
### University of Oklahoma
This notebook is the first of two (or three for graduate students) of the final project for Ethics and Compliance in AI at the University of Oklahoma. In this assignment, you will analyze a dataset containing credit card default information. The dataset was obtained from the UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients. It was initially used for the study Yeh, I., & Lien, C. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. _Expert Syst. Appl._, 36, 2473-2480.

The variables in the dataset are:
- ID: unique identifier
- LIMIT_BAL: amount of credit given in NT dollars
- SEX: gender of the client (1=Male, 2=Female)
- EDUCATION: education level (1=Graduate School, 2=University, 3=High School, 4=Others, 5=Unknown, 6=Unknown)
- MARRIAGE: marital status (1=Married, 2=Single, 3=Others)
- AGE: age in of client in years
- PAY_0 to PAY_6: monthly repayment status from September 2005 to April 2005 in reverse order. Note that there is no PAY_1. This is a naming mistake by the authors. (-2=No consumption, -1=Paid in full, 0=Minimum payment, 1-8=Payment delay for x months, 9=Payment delay for nine or more months)
- BILL_AMT1 to BILL_AMT6: amount of bill statement in NT dollars from September 2005 to April 2005 in reverse order
- PAY_AMT1 to PAY_AMT6: amount of previous payment in NT dollars from September 2005 to April 2005 in reverse order
- default payment next month: whether the client made the next month payment - this is the label variable (1=Defaulted, 0=No Default)

In this notebook, _final\_project\_part\_1.ipynb_, you will read in the credit dataset and explore it using fairness EDA techniques. You will also check missingness, determine if missingness is MCAR, and impute missing values. All methods you will need can be found in your homework assignments from the EDA and imputation modules.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
import seaborn as sns
import random

random.seed(42)

# Read in the dataset and perform initial cleaning and preprocessing
dataset = pd.read_csv('default_of_credit_card_clients.csv')
dataset.rename(columns={'default payment next month': 'DEFAULTED'}, inplace=True)
dataset.rename(columns={'PAY_0': 'PAY_1'}, inplace=True)  # Rename PAY_0 to PAY_1 for consistency
dataset.drop(columns=['ID'], inplace=True)  # Drop ID column as it is not useful for modeling

# Some fields contain codes that are undefined and should be treated as missing values
dataset['EDUCATION'] = dataset['EDUCATION'].replace([0, 5, 6], np.nan)
dataset['MARRIAGE'] = dataset['MARRIAGE'].replace(0, np.nan)

# Convert EDUCATION to a readable categorical variable
dataset['EDUCATION'] = dataset['EDUCATION'].astype('category')

#dataset['EDUCATION'] = dataset['EDUCATION'].map(education_mapping).astype('category')

# Convert MARRIAGE to a readable categorical variable
dataset['MARRIAGE'] = dataset['MARRIAGE'].astype('category')

#dataset['MARRIAGE'] = dataset['MARRIAGE'].map(marriage_mapping).astype('category')

# Convert SEX to a readable categorical variable
dataset['SEX'] = dataset['SEX'].astype('category')
sex_remapping = {
    1: 0,
    2: 1
}
dataset['SEX'] = dataset['SEX'].map(sex_remapping).astype('category')

# Convert age, credit limit, billing and payment amounts to float
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
dataset[['AGE', 'LIMIT_BAL'] + bill_cols + pay_cols] = dataset[['AGE', 'LIMIT_BAL'] + bill_cols + pay_cols].astype(float)

print("Fields and types:")
print(dataset.dtypes)

# Get missing value counts
print("Missing value counts:")
missing_counts = dataset.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(missing_counts)
print(f"\nTotal missing values: {dataset.isnull().sum().sum()}")

# Make lists of variables by type. This will be useful for imputation, scaling, and EDA.
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]
sensitive_vars = ['SEX']
categorical_vars = ['EDUCATION', 'MARRIAGE']
ordinal_vars = [f'PAY_{i}' for i in range(1, 7)]
numerical_vars = ['LIMIT_BAL', 'AGE'] + bill_cols + pay_cols

# Create dictionaries to translate various codes to more meaningful labels
sex_mapping = {
    0: 'male',
    1: 'female'
}
education_mapping = {
    1: 'graduate_school',
    2: 'university',
    3: 'high_school',
    4: 'other'
}
marriage_mapping = {
    1: 'married',
    2: 'single',
    3: 'other'
}
pay_mapping = {
    -2: 'No consumption',
    -1: 'Paid in full',
    0: 'Use revolving credit',
    1: 'Payment delay for one month',
    2: 'Payment delay for two months',
    3: 'Payment delay for three months',
    4: 'Payment delay for four months',
    5: 'Payment delay for five months',
    6: 'Payment delay for six months',
    7: 'Payment delay for seven months',
    8: 'Payment delay for eight months',
    9: 'Payment delay for nine months or more',
    np.nan: 'Unknown'
}

### Conduct EDA
Conduct EDA on the dataset.  You will conduct the following:
+ Stratified descriptive statistics
+ Distribution plots and Chi-Square analysis on categorical variables
+ Distribution plots and K-S distribution checks on continuous variables
+ Missingness analysis: MCAR, MAR, or MNAR
+ Association heatmap
+ Preliminary regression to determine if outcome depends on sensitive attribute



### Stratified descriptive statistics
We don't need to exampine _all_ the numeric variables. We will assume that checking BILL_AMT1, and PAY_AMT1 will suffice for all other features in those families. We will check LIMIT_BAL and AGE as well. In the section below, run simple descriptive statistics for numerical variables.

In [ ]:
# BEGIN YOUR CODE HERE:


# END HERE


### Categorical Variable Analysis
Plot the distributions of the categorical variables, including the treatment variable, versus the protected attribute. Then check to see if the distributions are significant with Chi-Square tests.

In [ ]:
# ENTER YOUR CODE FOR STACKED BAR PLOTS OF CATEGORICAL VARIABLES BY SENSITIVE ATTRIBUTE HERE
# YOU ONLY NEED ONE PAY VARIABLE TO SHOW PAYMENT DELAY DISTRIBUTION, SO USE PAY_1 FOR THIS PLOT

# BEGIN YOUR CODE HERE


# END HERE

In [ ]:
# CONDUCT CHI-SQUARE TESTS FOR CATEGORICAL VARIABLES HERE. HINT: CREATE A CROSSTAB
# AND PASS IT TO stats.chi2_contingency() TO GET THE TEST RESULTS

# BEGIN YOUR CODE HERE


# END HERE

### Numeric Variable Analysis
Plot the distributions of the numeric variables versus the protected attribute. Then check to see if the distributions are significant with K-S tests.

In [ ]:
# ENTER YOUR CODE FOR DISTRIBUTION PLOTS OF NUMERICAL VARIABLES BY SENSITIVE ATTRIBUTE HERE
# YOU ONLY NEED ONE PAY_AMT VARIABLE TO SHOW PAYMENT DELAY DISTRIBUTION, SO USE PAY_1 FOR THIS PLOT
# YOU ONLY NEED ONE BILL_AMT VARIABLE TO SHOW BILLING AMOUNT DISTRIBUTION, SO USE BILL_AMT1 FOR THIS PLOT

# BEGIN YOUR CODE HERE


# END HERE

In [ ]:
# CONDUCT KS-TESTS FOR NUMERICAL VARIABLES HERE

# BEGIN YOUR CODE HERE


# END HERE

### Missingness Analysis
We know from our general missingness analysis at the beginning of the script that EDUCATION and MARRIAGE have missing values. In this section determine whether any missingness is MCAR, MAR, or MNAR.

In [ ]:
# CHECK TO SEE IF MISSINGNESS IS MAR, MCAR, OR MNAR HERE. DONT FORGET TO Z-SCORE NORMALIZE
# YOUR NUMERICAL PREDICTORS AND ONE-HOT ENCODE YOUR CATEGORICAL PREDICTORS.

# BEGIN YOUR CODE HERE


# END HERE

### Correlation analyses
In this section, the code is provided for you because we did not demonstrate the use of Cramer's V. The next block calculates the Cramer's V statistic between combinations of categorical variables.

In [ ]:
def get_cramers_v(var1, var2):
    crosstab = pd.crosstab(var1, var2)
    chi2 = stats.chi2_contingency(crosstab)[0]
    n = crosstab.sum().sum()
    phi2 = chi2 / n
    r, k = crosstab.shape
    return np.sqrt(phi2 / min(k - 1, r - 1))

# Initialize an empty matrix
rows = []
vars = sensitive_vars + categorical_vars + ['DEFAULTED']
for var1 in vars:
    col = []
    for var2 in vars:
        value = get_cramers_v(dataset[var1], dataset[var2])
        col.append(value)
    rows.append(col)

cramers_matrix = pd.DataFrame(rows, columns=vars, index=vars)

plt.figure(figsize=(12, 10))
sns.heatmap(cramers_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Cramér\'s V Heatmap (Including Protected Attributes)')
plt.show()

### Correlation analyses continued

We again supply the code for a Pearson's correlation analysis of the sensitive attribute and the numerical values.

In [ ]:
cols_to_use = sensitive_vars + numerical_vars + ['DEFAULTED']
pearson_df = dataset[cols_to_use].dropna().copy()

# We need to convert SEX back to numeric for Pearson correlation
pearson_df['SEX']=pearson_df['SEX'].replace({'male': 0, 'female': 1})


# 2. Generate Pearson Correlation Matrix
corr_matrix = pearson_df.corr(method='pearson')

# 6. Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Including Protected Attributes)')
plt.show()

### Impute missing variables

Now that you have explored the dataset with respect to fairness, you need to impute missing variables and save the dataset to a CSV for use in the next portion of your analysis. Since the missing variables are categorical, you will use a function that efficiently imputes them using MICE. Use the code from your imputation assignment to impute the missing values.

In [ ]:
# IMPUTE MISSNG VALUES FOR EDUCATION AND MARRIAGE USING MICE IMPUTER HERE

# AN IMPUTATION FUNCTION IS PROVIDED FOR YOU HERE.

def impute_failsafe(df, target_cols=['EDUCATION', 'MARRIAGE'], label='DEFAULTED'):
    # 1. Feature selection and strict type casting
    X = df.drop(columns=[label]) if label in df.columns else df.copy()
    
    # Ensure all columns are numeric floats to prevent internal categorical handling
    X = X.astype('float32') 

    # 2. Use BayesianRidge (The most memory-efficient estimator)
    # This treats everything as a regression problem
    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        random_state=42,
        max_iter=10,
        verbose=2 
    )

    # 3. Fit and Transform
    imputed_array = imputer.fit_transform(X)
    
    # 4. Map back and ROUND
    imputed_df = pd.DataFrame(imputed_array, columns=X.columns, index=df.index)
    
    df_final = df.copy()
    for col in target_cols:
        # We round to the nearest integer because BayesianRidge produces floats
        df_final[col] = np.round(imputed_df[col]).astype(int)

    return df_final


In [ ]:
# CREATE A NEW DATAFRAME CALLED df_imputed FROM THE ORIGINAL dataset USING THE impute_failsafe FUNCTION

# BEGIN YOUR CODE HERE


# END HERE

In [ ]:
# ENSURE YOU HAVE THE SAME NUMBER OF ROWS BEFORE AND AFTER IMPUTATION
print(f"Original dataset shape: {dataset.shape}")
print(f"Imputed dataset shape: {df_imputed.shape}")

# CHECK TO SEE IF MISSINGNESS IS RESOLVED
print("\nMissing value counts after imputation:")
missing_counts_after = df_imputed.isnull().sum()
missing_counts_after = missing_counts_after[missing_counts_after > 0].sort_values(ascending=False)
print(missing_counts_after)
print(f"\nTotal missing values after imputation: {df_imputed.isnull().sum().sum()}")

In [ ]:
# CHECK TO SEE IF THE DISTRIBUTION OF THE IMPUTED VARIABLES IS SIMILAR TO THE ORIGINAL VARIABLES.

def check_distribution(original_df, imputed_df, column):
    print(f"--- Distribution Check: {column} ---")
    
    # Calculate percentages for both
    original_dist = original_df[column].value_counts(normalize=True).sort_index()
    imputed_dist = imputed_df[column].value_counts(normalize=True).sort_index()
    
    # Combine into a single table for easy comparison
    comparison = pd.DataFrame({
        'Original %': original_dist,
        'Imputed %': imputed_dist,
        'Difference': imputed_dist - original_dist
    })
    
    print(comparison)
    print("\n")

# Run for your imputed variables
check_distribution(dataset, df_imputed, 'EDUCATION')
check_distribution(dataset, df_imputed, 'MARRIAGE')

### Save your dataset

Save the dataset df_imputed to a CSV file for use in the next section of the Final Project

In [ ]:
# The modeling portion of the project will use this new imputed dataset. But the modeling functions
# don't like to work with category variables, so we will change SEX back to an integer.

df_imputed['SEX'] = df_imputed['SEX'].astype(int)  


In [ ]:
# SAVE df_imputed TO A NEW CSV FILE CALLED 'imputed_credit_card_data.csv'

# BEGIN YOUR CODE HERE


# END HERE